# 📘 Medallion Architecture Implementation
## Databricks Notebook — Bronze / Silver / Gold Mastery

**What you'll learn:**
- Medallion architecture theory and design principles
- Bronze: append-only raw ingestion with audit trails
- Silver: deduplication, cleaning, CDC processing, quality gates
- Gold: business aggregates, KPIs, dashboards
- Incremental processing, Delta Lake features
- NEW FinTech dataset with CDC simulation

**Prerequisites:** Notebooks 01-03 (techmart_dw, healthcare_dw)

---

---
# 🏗️ Setup & FinTech Dataset Creation

In [0]:
spark.sql("USE techmart_dw")
print(f"✅ techmart_dw: {spark.sql('SELECT COUNT(*) FROM orders').collect()[0][0]:,} orders")

In [0]:
%sql
CREATE DATABASE IF NOT EXISTS fintech_dw;
USE fintech_dw;

In [0]:
%sql
-- Accounts table (5000 rows)
DROP TABLE IF EXISTS accounts;
CREATE TABLE accounts (
  account_id INT, customer_name STRING, account_type STRING, balance DECIMAL(14,2),
  currency STRING, opened_date DATE, status STRING, branch_id INT, risk_score INT
);
INSERT INTO accounts
SELECT id AS account_id,
  CONCAT('Customer_', id) AS customer_name,
  CASE abs(hash(id*3))%3 WHEN 0 THEN 'checking' WHEN 1 THEN 'savings' ELSE 'investment' END AS account_type,
  CAST(abs(hash(id*7))%500000+100 AS DECIMAL(14,2)) AS balance,
  CASE abs(hash(id*11))%4 WHEN 0 THEN 'USD' WHEN 1 THEN 'EUR' WHEN 2 THEN 'GBP' ELSE 'USD' END AS currency,
  date_add('2015-01-01', abs(hash(id*13))%3000) AS opened_date,
  CASE abs(hash(id*17))%5 WHEN 0 THEN 'active' WHEN 1 THEN 'active' WHEN 2 THEN 'active' WHEN 3 THEN 'frozen' ELSE 'closed' END AS status,
  abs(hash(id*19))%50+1 AS branch_id,
  abs(hash(id*23))%100+1 AS risk_score
FROM (SELECT explode(sequence(1, 5000)) AS id);

In [0]:
%sql
-- Wire Transfers with CDC markers (50000 rows)
DROP TABLE IF EXISTS wire_transfers;
CREATE TABLE wire_transfers (
  transfer_id INT, from_account INT, to_account INT, amount DECIMAL(12,2),
  currency STRING, transfer_date DATE, transfer_timestamp TIMESTAMP, status STRING,
  reference_number STRING, notes STRING, _cdc_operation STRING, _cdc_timestamp TIMESTAMP
);
INSERT INTO wire_transfers
SELECT id AS transfer_id,
  abs(hash(id*3))%5000+1 AS from_account,
  abs(hash(id*7))%5000+1 AS to_account,
  CAST(abs(hash(id*11))%100000+10 AS DECIMAL(12,2)) AS amount,
  CASE abs(hash(id*13))%3 WHEN 0 THEN 'USD' WHEN 1 THEN 'EUR' ELSE 'GBP' END AS currency,
  date_add('2023-01-01', abs(hash(id*17))%600) AS transfer_date,
  CAST(date_add('2023-01-01', abs(hash(id*17))%600) AS TIMESTAMP) AS transfer_timestamp,
  CASE abs(hash(id*19))%5 WHEN 0 THEN 'completed' WHEN 1 THEN 'pending' WHEN 2 THEN 'failed' WHEN 3 THEN 'reversed' ELSE 'completed' END AS status,
  CONCAT('REF-', LPAD(CAST(id AS STRING), 8, '0')) AS reference_number,
  CASE WHEN id%20=0 THEN 'Flagged for review' ELSE NULL END AS notes,
  CASE abs(hash(id*23))%10 WHEN 0 THEN 'U' WHEN 1 THEN 'D' ELSE 'I' END AS _cdc_operation,
  CAST(date_add('2023-01-01', abs(hash(id*17))%600) AS TIMESTAMP) AS _cdc_timestamp
FROM (SELECT explode(sequence(1, 50000)) AS id);

In [0]:
%sql
-- Fraud Signals (10000 rows)
DROP TABLE IF EXISTS fraud_signals;
CREATE TABLE fraud_signals (
  signal_id INT, account_id INT, signal_type STRING, severity STRING,
  detected_at TIMESTAMP, details_json STRING, is_confirmed BOOLEAN, reviewed_by STRING
);
INSERT INTO fraud_signals
SELECT id AS signal_id,
  abs(hash(id*3))%5000+1 AS account_id,
  CASE abs(hash(id*7))%6 WHEN 0 THEN 'velocity_spike' WHEN 1 THEN 'geo_anomaly' WHEN 2 THEN 'amount_outlier' WHEN 3 THEN 'pattern_match' WHEN 4 THEN 'device_change' ELSE 'time_anomaly' END AS signal_type,
  CASE abs(hash(id*11))%4 WHEN 0 THEN 'critical' WHEN 1 THEN 'high' WHEN 2 THEN 'medium' ELSE 'low' END AS severity,
  CAST(date_add('2023-06-01', abs(hash(id*13))%365) AS TIMESTAMP) AS detected_at,
  CONCAT('{"score":', abs(hash(id*17))%100, ',"rule":"rule_', abs(hash(id*19))%20, '"}') AS details_json,
  CASE WHEN abs(hash(id*23))%4=0 THEN true ELSE false END AS is_confirmed,
  CASE WHEN abs(hash(id*29))%3=0 THEN CONCAT('analyst_', abs(hash(id*31))%10) ELSE NULL END AS reviewed_by
FROM (SELECT explode(sequence(1, 10000)) AS id);

In [0]:
%sql
-- KYC Records (5000 rows)
DROP TABLE IF EXISTS kyc_records;
CREATE TABLE kyc_records (
  kyc_id INT, account_id INT, verification_type STRING, status STRING,
  submitted_date DATE, approved_date DATE, document_type STRING, risk_level STRING
);
INSERT INTO kyc_records
SELECT id AS kyc_id,
  abs(hash(id*3))%5000+1 AS account_id,
  CASE abs(hash(id*7))%3 WHEN 0 THEN 'identity' WHEN 1 THEN 'address' ELSE 'income' END AS verification_type,
  CASE abs(hash(id*11))%4 WHEN 0 THEN 'approved' WHEN 1 THEN 'pending' WHEN 2 THEN 'rejected' ELSE 'expired' END AS status,
  date_add('2022-01-01', abs(hash(id*13))%900) AS submitted_date,
  CASE WHEN abs(hash(id*11))%4=0 THEN date_add('2022-01-01', abs(hash(id*13))%900+abs(hash(id*17))%30) ELSE NULL END AS approved_date,
  CASE abs(hash(id*19))%4 WHEN 0 THEN 'passport' WHEN 1 THEN 'drivers_license' WHEN 2 THEN 'utility_bill' ELSE 'bank_statement' END AS document_type,
  CASE abs(hash(id*23))%3 WHEN 0 THEN 'low' WHEN 1 THEN 'medium' ELSE 'high' END AS risk_level
FROM (SELECT explode(sequence(1, 5000)) AS id);

In [0]:
%sql
-- Compliance Events (20000 rows)
DROP TABLE IF EXISTS compliance_events;
CREATE TABLE compliance_events (
  event_id INT, account_id INT, event_type STRING, event_date DATE,
  amount_involved DECIMAL(12,2), regulatory_body STRING, resolution_status STRING
);
INSERT INTO compliance_events
SELECT id AS event_id,
  abs(hash(id*3))%5000+1 AS account_id,
  CASE abs(hash(id*7))%5 WHEN 0 THEN 'SAR_filed' WHEN 1 THEN 'CTR_filed' WHEN 2 THEN 'AML_alert' WHEN 3 THEN 'sanctions_check' ELSE 'PEP_screening' END AS event_type,
  date_add('2023-01-01', abs(hash(id*11))%600) AS event_date,
  CAST(abs(hash(id*13))%200000+1000 AS DECIMAL(12,2)) AS amount_involved,
  CASE abs(hash(id*17))%3 WHEN 0 THEN 'FinCEN' WHEN 1 THEN 'OCC' ELSE 'FDIC' END AS regulatory_body,
  CASE abs(hash(id*19))%4 WHEN 0 THEN 'resolved' WHEN 1 THEN 'open' WHEN 2 THEN 'escalated' ELSE 'dismissed' END AS resolution_status
FROM (SELECT explode(sequence(1, 20000)) AS id);

In [0]:
%sql
SELECT 'accounts' AS tbl, COUNT(*) AS rows FROM accounts
UNION ALL SELECT 'wire_transfers', COUNT(*) FROM wire_transfers
UNION ALL SELECT 'fraud_signals', COUNT(*) FROM fraud_signals
UNION ALL SELECT 'kyc_records', COUNT(*) FROM kyc_records
UNION ALL SELECT 'compliance_events', COUNT(*) FROM compliance_events
ORDER BY tbl;

---
# 📗 Section 1: Medallion Architecture Theory

## What Is It?

```
┌──────────────────────────────────────────────────────────────────┐
│                    MEDALLION ARCHITECTURE                          │
├──────────────┬──────────────────┬────────────────────────────────┤
│   🥉 BRONZE   │    🥈 SILVER      │         🥇 GOLD                │
│              │                  │                                │
│  Raw Data    │  Cleaned Data    │  Business Aggregates           │
│  As-Is       │  Validated       │  KPIs & Metrics                │
│  Append-Only │  Deduplicated    │  BI-Ready                      │
│  Full Audit  │  Standardized    │  Pre-Computed                  │
│              │                  │                                │
│  Schema:     │  Schema:         │  Schema:                       │
│  on-read     │  enforced        │  star/snowflake                │
│              │                  │                                │
│  Consumers:  │  Consumers:      │  Consumers:                    │
│  Data Eng    │  Data Scientists │  Business Analysts             │
│  Debugging   │  ML Engineers    │  Dashboards                    │
└──────────────┴──────────────────┴────────────────────────────────┘
```

## Why It Exists

1. **Separation of concerns** — each layer has one job
2. **Reprocessing** — if Silver logic changes, Bronze is untouched
3. **Debugging** — raw data always available for investigation
4. **Quality gates** — validate between layers
5. **Performance** — Gold is pre-aggregated for fast queries

## When to Skip a Layer

- **Skip Bronze:** Only for ephemeral streaming data you'll never replay
- **Skip Silver:** Only for simple aggregations on already-clean source data
- **Never skip Gold:** If anyone queries the data, they need a Gold layer

## Why Medallion? The Problems It Solves

### Without Medallion (the "Data Swamp"):
```
Source → ??? → Analytics
         │
         └── No clear layers
         └── Raw and clean data mixed together
         └── Can't reprocess (raw data modified)
         └── No audit trail
         └── Quality issues propagate silently
```

### With Medallion:
```
Source → Bronze (raw) → Silver (clean) → Gold (business)
         │                │                │
         └── Immutable    └── Validated    └── Aggregated
         └── Audit trail  └── Typed        └── Ready for BI
         └── Reprocessable└── Deduplicated └── Ready for ML
```

## Layer Responsibilities — Detailed

| Aspect | Bronze | Silver | Gold |
|--------|--------|--------|------|
| **Data quality** | As-is (garbage included) | Validated, cleaned | Business-rule verified |
| **Schema** | Schema-on-read (flexible) | Schema enforced (typed) | Schema enforced (optimized) |
| **Duplicates** | Allowed (append-only) | Removed (deduplicated) | N/A (aggregated) |
| **NULLs** | Preserved | Handled (default/filter) | No unexpected NULLs |
| **Format** | Source format (JSON, CSV, etc.) | Delta Lake (Parquet) | Delta Lake (Parquet) |
| **Retention** | Long (years) | Medium (months) | Short (weeks/months) |
| **Access** | Data engineers only | Engineers + scientists | Everyone (analysts, BI) |
| **Updates** | Append-only (never modify!) | MERGE (upsert) | Overwrite or MERGE |
| **Partitioning** | By ingestion date | By business date | By query pattern |

## Real-World Medallion Examples

### Netflix
- **Bronze**: Raw viewing events from 200M+ users (append-only, Parquet on S3)
- **Silver**: Cleaned events (deduplicated, validated user_id, enriched with content metadata)
- **Gold**: Daily viewing metrics, recommendation features, A/B test results

### Uber
- **Bronze**: Raw trip events from Kafka (GPS pings, ride requests, completions)
- **Silver**: Cleaned trips (matched rider↔driver, validated coordinates, calculated distance)
- **Gold**: City-level metrics, driver earnings, surge pricing inputs

### Your Company (TechMart)
- **Bronze**: Raw orders, customers, payments from source systems
- **Silver**: Clean orders (validated amounts, standardized status, deduplicated)
- **Gold**: Daily revenue, customer segments, product performance

In [0]:
# Medallion architecture decision framework
def medallion_layer_decision(data_characteristic):
    """Decide what processing belongs in which layer."""
    
    bronze_tasks = [
        "Land raw data exactly as received",
        "Add ingestion metadata (_ingested_at, _source_file, _batch_id)",
        "Append-only (never update or delete!)",
        "Minimal schema (schema-on-read)",
        "Keep ALL records (even bad ones)",
    ]
    
    silver_tasks = [
        "Deduplicate records (ROW_NUMBER pattern)",
        "Cast types (string dates → DATE, string amounts → DECIMAL)",
        "Validate required fields (reject NULLs in key columns)",
        "Standardize values (lowercase status, trim whitespace)",
        "Enrich with reference data (join with dim tables)",
        "Apply business rules (filter test records, invalid amounts)",
        "Handle late-arriving data (MERGE with watermark)",
    ]
    
    gold_tasks = [
        "Aggregate for business use (daily revenue, monthly cohorts)",
        "Join multiple silver tables (customer 360, order details)",
        "Calculate derived metrics (LTV, churn risk, conversion rate)",
        "Optimize for query patterns (partition by date, cluster by region)",
        "Pre-compute expensive calculations (avoid repeated full scans)",
    ]
    
    print("📊 MEDALLION LAYER RESPONSIBILITIES")
    print("=" * 60)
    print("\n🥉 BRONZE (Raw Landing):")
    for task in bronze_tasks:
        print(f"   • {task}")
    print("\n🥈 SILVER (Cleaned & Validated):")
    for task in silver_tasks:
        print(f"   • {task}")
    print("\n🥇 GOLD (Business-Ready):")
    for task in gold_tasks:
        print(f"   • {task}")
    
    print("\n⚠️ COMMON MISTAKES:")
    mistakes = [
        "❌ Modifying Bronze data (breaks reprocessability!)",
        "❌ Putting business logic in Bronze (too early)",
        "❌ Skipping Silver (raw → gold = fragile)",
        "❌ One Gold table for everything (becomes a bottleneck)",
        "❌ No quality checks between layers (bad data propagates)",
    ]
    for m in mistakes:
        print(f"   {m}")

medallion_layer_decision(None)


---
# 📗 Section 2: Bronze Layer Implementation

## Design Principles
- **Append-only:** Never update or delete Bronze records
- **No transformations:** Store exactly what arrived
- **Full audit:** Who, when, where, what batch
- **Schema-on-read:** Accept whatever schema arrives

In [0]:
from pyspark.sql.functions import *

# Bronze ingestion function
def ingest_to_bronze(source_table, bronze_table, source_system, batch_id):
    """Ingest a source table into bronze with audit columns."""
    df = spark.table(source_table)
    
    bronze_df = (df
        .withColumn("_ingested_at", current_timestamp())
        .withColumn("_source_system", lit(source_system))
        .withColumn("_batch_id", lit(batch_id))
        .withColumn("_file_name", lit(f"{source_table}_extract"))
    )
    
    bronze_df.write.format("delta").mode("overwrite").saveAsTable(bronze_table)
    count = spark.table(bronze_table).count()
    print(f"✅ {bronze_table}: {count:,} rows ingested")
    return count

In [0]:
# Ingest TechMart tables into bronze
spark.sql("USE fintech_dw")

ingest_to_bronze("techmart_dw.orders", "bronze_orders", "techmart_api", "batch_20240601")
ingest_to_bronze("techmart_dw.payments", "bronze_payments", "payment_gateway", "batch_20240601")
ingest_to_bronze("techmart_dw.customers", "bronze_customers", "crm_system", "batch_20240601")

In [0]:
# Ingest FinTech tables into bronze
ingest_to_bronze("fintech_dw.wire_transfers", "bronze_wire_transfers", "core_banking", "batch_20240601")
ingest_to_bronze("fintech_dw.fraud_signals", "bronze_fraud_signals", "fraud_engine", "batch_20240601")
ingest_to_bronze("fintech_dw.compliance_events", "bronze_compliance_events", "compliance_system", "batch_20240601")

In [0]:
# Verify bronze layer
spark.sql("USE fintech_dw")
bronze_df = spark.table("bronze_wire_transfers")
print("Bronze wire_transfers schema (note audit columns):")
bronze_df.printSchema()
bronze_df.select("transfer_id", "amount", "_ingested_at", "_source_system", "_batch_id").show(5)

In [0]:
# ============================================
# 🎯 YOUR TURN: Bronze Ingestion
# ============================================
# Ingest fintech_dw.kyc_records into a bronze table called "bronze_kyc"
# Add audit columns: _ingested_at, _source_system='kyc_portal', _batch_id='batch_20240601'
# Verify the row count and show the schema
#
# Write your code below:


In [0]:
# ============================================
# ✅ SOLUTION
# ============================================
ingest_to_bronze("fintech_dw.kyc_records", "bronze_kyc", "kyc_portal", "batch_20240601")
spark.table("bronze_kyc").printSchema()

## Bronze Layer Best Practices

### The Append-Only Rule

> ⚠️ **NEVER modify Bronze data.** If your transformation has a bug,
> you need the raw data to reprocess. Bronze is your safety net.

### Metadata Columns (Add to EVERY Bronze table)

```sql
CREATE TABLE bronze.raw_orders (
    -- Source columns (as-is)
    order_id STRING,        -- Keep as STRING even if it's a number!
    amount STRING,          -- Don't cast yet — might have "N/A" values
    status STRING,
    order_date STRING,      -- Source might send "2024/03/15" or "March 15, 2024"
    
    -- Metadata columns (added by your pipeline)
    _ingested_at TIMESTAMP,     -- When did we ingest this?
    _source_file STRING,        -- Which file did it come from?
    _batch_id STRING,           -- Which batch/run?
    _source_system STRING,      -- Which source system?
    _row_hash STRING            -- MD5 hash for change detection
) USING DELTA
PARTITIONED BY (_ingested_at)   -- Partition by ingestion date for easy cleanup
```

### Multiple Source Formats

In production, Bronze handles MANY formats:

In [0]:
# Bronze ingestion patterns for different source formats
print("📥 BRONZE INGESTION PATTERNS")
print("=" * 60)

# Pattern 1: JSON from APIs
print("""
1️⃣ JSON from REST APIs:
   spark.read.format("json")
       .option("multiLine", True)
       .option("mode", "PERMISSIVE")  # Don't fail on bad records
       .option("columnNameOfCorruptRecord", "_corrupt_record")
       .load("/raw/api_responses/2024-03-15/")
""")

# Pattern 2: CSV from SFTP/file drops
print("""
2️⃣ CSV from file drops:
   spark.read.format("csv")
       .option("header", True)
       .option("inferSchema", False)  # Keep everything as STRING in Bronze!
       .option("mode", "PERMISSIVE")
       .load("/raw/sftp_drops/orders/")
""")

# Pattern 3: Streaming from Kafka
print("""
3️⃣ Streaming from Kafka:
   spark.readStream.format("kafka")
       .option("kafka.bootstrap.servers", "broker:9092")
       .option("subscribe", "orders-topic")
       .option("startingOffsets", "earliest")
       .load()
       .selectExpr("CAST(value AS STRING)", "timestamp", "topic", "partition", "offset")
""")

# Pattern 4: Auto Loader (recommended for files)
print("""
4️⃣ Auto Loader (RECOMMENDED for file ingestion):
   spark.readStream.format("cloudFiles")
       .option("cloudFiles.format", "json")
       .option("cloudFiles.schemaLocation", "/checkpoints/orders/schema")
       .option("cloudFiles.inferColumnTypes", True)
       .load("/raw/orders/")
""")

print("💡 Key Principle: In Bronze, keep data AS-IS. Don't transform yet!")

---
# 📗 Section 3: Silver Layer Implementation

## 3.1 Deduplication Patterns

In [0]:
from pyspark.sql.functions import *
from pyspark.sql.window import Window

# Deduplicate bronze orders (ROW_NUMBER pattern)
bronze_orders = spark.table("bronze_orders")
print(f"Bronze orders: {bronze_orders.count():,}")

w = Window.partitionBy("order_id").orderBy(col("_ingested_at").desc())
deduped = (bronze_orders
    .withColumn("_rn", row_number().over(w))
    .filter("_rn = 1")
    .drop("_rn")
)
print(f"After dedup: {deduped.count():,}")

## 3.2 Data Cleaning

In [0]:
# Silver: clean and standardize orders
silver_orders = (deduped
    # Standardize strings
    .withColumn("status", lower(trim(col("status"))))
    .withColumn("payment_method", lower(trim(col("payment_method"))))
    .withColumn("order_source", lower(trim(col("order_source"))))
    # Fix negative amounts
    .withColumn("total_amount", when(col("total_amount") < 0, lit(0)).otherwise(col("total_amount")))
    # Add derived columns
    .withColumn("net_amount", col("total_amount") - col("discount_amount"))
    .withColumn("order_year", year("order_date"))
    .withColumn("order_month", month("order_date"))
    # Null handling
    .withColumn("notes", coalesce(col("notes"), lit("")))
    # Quality flag
    .withColumn("_is_valid", 
        (col("order_id").isNotNull()) & 
        (col("customer_id").isNotNull()) & 
        (col("total_amount") >= 0)
    )
    # Audit
    .withColumn("_processed_at", current_timestamp())
    .withColumn("_row_hash", md5(concat_ws("|",
        col("order_id").cast("string"), col("total_amount").cast("string"), col("status"))))
)

silver_orders.write.format("delta").mode("overwrite").saveAsTable("silver_orders")
print(f"Silver orders: {spark.table('silver_orders').count():,}")
print(f"Invalid records: {silver_orders.filter('_is_valid = false').count()}")

## 3.3 Validation & Quality Gates

In [0]:
# Quality gate: check rules before promoting to silver
def quality_gate(df, table_name, rules):
    """Run quality checks and return pass/fail with metrics."""
    total = df.count()
    results = []
    
    for rule_name, condition in rules.items():
        failures = df.filter(f"NOT ({condition})").count()
        pass_rate = (total - failures) / total * 100
        results.append({
            "rule": rule_name,
            "total": total,
            "failures": failures,
            "pass_rate": round(pass_rate, 2),
            "status": "PASS" if pass_rate >= 95 else "FAIL"
        })
    
    print(f"\n{'='*60}")
    print(f"Quality Gate: {table_name}")
    print(f"{'='*60}")
    all_pass = True
    for r in results:
        icon = "✅" if r["status"] == "PASS" else "❌"
        print(f"  {icon} {r['rule']}: {r['pass_rate']}% ({r['failures']} failures)")
        if r["status"] == "FAIL":
            all_pass = False
    
    gate_status = "PASSED" if all_pass else "FAILED"
    print(f"\n  Gate Status: {'🟢' if all_pass else '🔴'} {gate_status}")
    print(f"{'='*60}")
    return all_pass, results

# Run quality gate on bronze payments
bronze_payments = spark.table("bronze_payments")
rules = {
    "payment_id_not_null": "payment_id IS NOT NULL",
    "amount_positive": "amount > 0",
    "valid_status": "payment_status IN ('completed','pending','failed','refunded')",
    "date_not_future": "payment_date <= current_date()"
}
passed, metrics = quality_gate(bronze_payments, "bronze_payments", rules)

## 3.4 Quarantine Pattern

In [0]:
# Quarantine: separate good from bad records
bronze_payments = spark.table("bronze_payments")

# Split into valid and quarantine
valid_payments = bronze_payments.filter("payment_id IS NOT NULL AND amount > 0")
quarantine_payments = bronze_payments.filter("payment_id IS NULL OR amount <= 0")

valid_payments.write.format("delta").mode("overwrite").saveAsTable("silver_payments")
quarantine_payments.write.format("delta").mode("overwrite").saveAsTable("quarantine_payments")

print(f"Valid → silver_payments: {spark.table('silver_payments').count():,}")
print(f"Quarantine: {spark.table('quarantine_payments').count():,}")

## 3.5 CDC Processing

In [0]:
# CDC Processing: Apply INSERT/UPDATE/DELETE from wire_transfers
bronze_wt = spark.table("bronze_wire_transfers")

# Step 1: Deduplicate CDC events (keep latest per transfer_id)
w = Window.partitionBy("transfer_id").orderBy(col("_cdc_timestamp").desc())
latest_cdc = (bronze_wt
    .withColumn("_rn", row_number().over(w))
    .filter("_rn = 1")
    .drop("_rn")
)

# Step 2: Separate by operation
inserts = latest_cdc.filter("_cdc_operation = 'I'")
updates = latest_cdc.filter("_cdc_operation = 'U'")
deletes = latest_cdc.filter("_cdc_operation = 'D'")

print(f"CDC breakdown:")
print(f"  Inserts: {inserts.count():,}")
print(f"  Updates: {updates.count():,}")
print(f"  Deletes: {deletes.count():,}")

# Step 3: Build silver (exclude deletes, apply latest state)
silver_wt = (latest_cdc
    .filter("_cdc_operation != 'D'")
    .withColumn("status", lower(trim(col("status"))))
    .withColumn("_processed_at", current_timestamp())
)

silver_wt.write.format("delta").mode("overwrite").saveAsTable("silver_wire_transfers")
print(f"\nSilver wire_transfers: {spark.table('silver_wire_transfers').count():,}")

## 3.6 MERGE for CDC Application

In [0]:
%sql
-- MERGE pattern for applying CDC incrementally
-- First create the target silver table
DROP TABLE IF EXISTS silver_wire_transfers_v2;
CREATE TABLE silver_wire_transfers_v2 AS
SELECT transfer_id, from_account, to_account, amount, currency,
  transfer_date, status, reference_number, _cdc_timestamp, current_timestamp() AS _processed_at
FROM bronze_wire_transfers
WHERE _cdc_operation = 'I'
LIMIT 10000;

In [0]:
%sql
-- Apply CDC batch using MERGE
MERGE INTO silver_wire_transfers_v2 AS target
USING (
  SELECT * FROM (
    SELECT *, ROW_NUMBER() OVER (PARTITION BY transfer_id ORDER BY _cdc_timestamp DESC) AS rn
    FROM bronze_wire_transfers
    WHERE _cdc_operation IN ('I', 'U')
  ) WHERE rn = 1
) AS source
ON target.transfer_id = source.transfer_id
WHEN MATCHED AND source._cdc_timestamp > target._cdc_timestamp THEN
  UPDATE SET
    target.amount = source.amount,
    target.status = source.status,
    target._cdc_timestamp = source._cdc_timestamp,
    target._processed_at = current_timestamp()
WHEN NOT MATCHED THEN
  INSERT (transfer_id, from_account, to_account, amount, currency, transfer_date, status, reference_number, _cdc_timestamp, _processed_at)
  VALUES (source.transfer_id, source.from_account, source.to_account, source.amount, source.currency, source.transfer_date, source.status, source.reference_number, source._cdc_timestamp, current_timestamp());

In [0]:
# ============================================
# 🎯 YOUR TURN: Silver CDC Processing
# ============================================
# Process bronze_compliance_events into silver:
# 1. Deduplicate by event_id (keep latest _ingested_at)
# 2. Standardize: lowercase event_type and resolution_status
# 3. Validate: event_id NOT NULL, amount_involved > 0
# 4. Add _processed_at and _row_hash
# 5. Write to silver_compliance_events
# 6. Print row count
#
# Write your code below:


In [0]:
# ============================================
# ✅ SOLUTION
# ============================================
from pyspark.sql.functions import *
from pyspark.sql.window import Window

bronze_ce = spark.table("bronze_compliance_events")
w = Window.partitionBy("event_id").orderBy(col("_ingested_at").desc())

silver_ce = (bronze_ce
    .withColumn("_rn", row_number().over(w))
    .filter("_rn = 1")
    .drop("_rn")
    .withColumn("event_type", lower(trim(col("event_type"))))
    .withColumn("resolution_status", lower(trim(col("resolution_status"))))
    .filter("event_id IS NOT NULL AND amount_involved > 0")
    .withColumn("_processed_at", current_timestamp())
    .withColumn("_row_hash", md5(concat_ws("|",
        col("event_id").cast("string"), col("amount_involved").cast("string"), col("resolution_status"))))
)

silver_ce.write.format("delta").mode("overwrite").saveAsTable("silver_compliance_events")
print(f"Silver compliance_events: {spark.table('silver_compliance_events').count():,}")

## Silver Layer Patterns — Deep Dive

### The Deduplication Pattern (Most Common Silver Operation)

```sql
-- Pattern: Keep only the LATEST record per business key
-- Uses ROW_NUMBER to rank duplicates, keeps rank=1

WITH ranked AS (
    SELECT *,
        ROW_NUMBER() OVER (
            PARTITION BY order_id           -- Business key
            ORDER BY _ingested_at DESC      -- Latest wins
        ) AS rn
    FROM bronze.raw_orders
)
SELECT * FROM ranked WHERE rn = 1
```

### The Quarantine Pattern

Records that fail validation go to a quarantine table (not discarded!):

```
Bronze → Validation → Silver (good records)
                   └→ Quarantine (bad records) → Manual review → Fix → Re-inject
```

In [0]:
# Silver layer: Complete transformation pipeline
from pyspark.sql.functions import col, when, trim, lower, to_date, md5, concat_ws, current_timestamp, row_number, lit
from pyspark.sql.window import Window

print("🥈 SILVER LAYER — Complete Transformation Pipeline")
print("=" * 60)

# Simulate Bronze data (with quality issues)
bronze_data = spark.createDataFrame([
    ("1001", "42", "99.99", "COMPLETED", "2024-03-15", "2024-03-15 10:00:00", "batch_001"),
    ("1001", "42", "99.99", "COMPLETED", "2024-03-15", "2024-03-15 12:00:00", "batch_002"),  # DUPLICATE!
    ("1002", "17", "invalid", "pending", "2024-03-15", "2024-03-15 10:01:00", "batch_001"),  # BAD amount
    ("1003", None, "300.00", "shipped", "2024-03-16", "2024-03-16 08:00:00", "batch_003"),   # NULL customer
    ("1004", "99", "45.50", "  Completed  ", "2024/03/16", "2024-03-16 08:01:00", "batch_003"),  # Messy status
    ("1005", "55", "-10.00", "completed", "2024-03-16", "2024-03-16 08:02:00", "batch_003"),  # Negative amount
], ["order_id", "customer_id", "amount", "status", "order_date", "_ingested_at", "_batch_id"])

print(f"Bronze records: {bronze_data.count()}")
bronze_data.show(truncate=False)

# Step 1: Deduplicate (keep latest per order_id)
window = Window.partitionBy("order_id").orderBy(col("_ingested_at").desc())
deduped = bronze_data.withColumn("rn", row_number().over(window)).filter(col("rn") == 1).drop("rn")
print(f"\nAfter deduplication: {deduped.count()} records")

# Step 2: Type casting and cleaning
cleaned = (deduped
    .withColumn("amount_clean", col("amount").cast("decimal(12,2)"))
    .withColumn("status_clean", lower(trim(col("status"))))
    .withColumn("customer_id_clean", col("customer_id").cast("int"))
)

# Step 3: Validation — split into good and quarantine
good_records = cleaned.filter(
    col("amount_clean").isNotNull() &
    (col("amount_clean") > 0) &
    col("customer_id_clean").isNotNull()
)

quarantine_records = cleaned.filter(
    col("amount_clean").isNull() |
    (col("amount_clean") <= 0) |
    col("customer_id_clean").isNull()
)

print(f"\n✅ Good records (→ Silver): {good_records.count()}")
print(f"❌ Quarantine records: {quarantine_records.count()}")

print("\nGood records:")
good_records.select("order_id", "customer_id_clean", "amount_clean", "status_clean").show()

print("Quarantine records (need manual review):")
quarantine_records.select("order_id", "customer_id", "amount", "status").show()


In [0]:
# ============================================
# 🎯 YOUR TURN: Build a Silver Transformation
# ============================================
# Given the techmart_dw.orders table (Bronze-like), create a Silver version:
#
# 1. Deduplicate on order_id (keep latest by created_at)
# 2. Cast total_amount to DECIMAL(12,2)
# 3. Standardize status (lowercase, trim)
# 4. Filter out: NULL customer_id, negative amounts, status='test'
# 5. Add _processed_at timestamp
# 6. Add _row_hash (MD5 of key columns for change detection)
#
# Write your transformation below:


In [0]:
# ============================================
# ✅ SOLUTION
# ============================================
from pyspark.sql.functions import col, lower, trim, current_timestamp, md5, concat_ws, row_number
from pyspark.sql.window import Window

# Read Bronze
bronze = spark.table("techmart_dw.orders")

# Deduplicate
window = Window.partitionBy("order_id").orderBy(col("created_at").desc())
deduped = bronze.withColumn("rn", row_number().over(window)).filter(col("rn") == 1).drop("rn")

# Clean and validate
silver_orders = (deduped
    # Standardize
    .withColumn("status", lower(trim(col("status"))))
    # Filter bad records
    .filter(col("customer_id").isNotNull())
    .filter(col("total_amount") > 0)
    .filter(col("status") != "test")
    # Add audit columns
    .withColumn("_processed_at", current_timestamp())
    .withColumn("_row_hash", md5(concat_ws("|", 
        col("order_id"), col("customer_id"), col("total_amount"), col("status"))))
)

print(f"Bronze: {bronze.count():,} rows")
print(f"Silver: {silver_orders.count():,} rows")
print(f"Removed: {bronze.count() - silver_orders.count():,} rows (duplicates + invalid)")
silver_orders.select("order_id", "customer_id", "total_amount", "status", "_processed_at", "_row_hash").show(5)

---
# 📗 Section 4: Gold Layer Implementation

## 4.1 Gold Design Principles
- Pre-computed for fast queries
- Business-friendly column names
- Aggregated at the right grain
- Optimized for BI tools

In [0]:
# Gold: Daily Sales Summary
from pyspark.sql.functions import *
from pyspark.sql.window import Window

silver_orders = spark.table("silver_orders")

gold_daily_sales = (silver_orders
    .filter("_is_valid = true AND status = 'completed'")
    .groupBy("order_date", "payment_method", "order_source")
    .agg(
        count("*").alias("order_count"),
        countDistinct("customer_id").alias("unique_customers"),
        round(sum("total_amount"), 2).alias("total_revenue"),
        round(avg("total_amount"), 2).alias("avg_order_value"),
        round(sum("discount_amount"), 2).alias("total_discounts"),
        round(sum("net_amount"), 2).alias("net_revenue")
    )
)

# Add running totals and moving averages
w_date = Window.orderBy("order_date").rowsBetween(-6, 0)
gold_daily_sales = (gold_daily_sales
    .withColumn("ma_7d_revenue", round(avg("total_revenue").over(w_date), 2))
    .withColumn("_computed_at", current_timestamp())
)

gold_daily_sales.write.format("delta").mode("overwrite").saveAsTable("gold_daily_sales")
print(f"Gold daily sales: {spark.table('gold_daily_sales').count():,} rows")
gold_daily_sales.orderBy(col("order_date").desc()).show(10)

## 4.2 Gold: Customer 360

In [0]:
# Gold: Customer 360 with engagement scoring
silver_orders = spark.table("silver_orders")
bronze_customers = spark.table("bronze_customers")

# Order metrics per customer
order_metrics = (silver_orders
    .filter("_is_valid = true")
    .groupBy("customer_id")
    .agg(
        count("*").alias("total_orders"),
        sum(when(col("status") == "completed", 1).otherwise(0)).alias("completed_orders"),
        round(sum("total_amount"), 2).alias("lifetime_revenue"),
        round(avg("total_amount"), 2).alias("avg_order_value"),
        min("order_date").alias("first_order_date"),
        max("order_date").alias("last_order_date"),
        countDistinct("payment_method").alias("payment_methods_used")
    )
    .withColumn("days_since_last_order", datediff(current_date(), col("last_order_date")))
    .withColumn("customer_tenure_days", datediff(col("last_order_date"), col("first_order_date")))
)

# Engagement score
gold_customer_360 = (order_metrics
    .withColumn("engagement_score",
        (col("total_orders") * 10) +
        (when(col("days_since_last_order") < 30, 50)
         .when(col("days_since_last_order") < 90, 30)
         .when(col("days_since_last_order") < 180, 10)
         .otherwise(0)) +
        (col("payment_methods_used") * 5)
    )
    .withColumn("churn_risk",
        when(col("days_since_last_order") > 365, "High")
        .when(col("days_since_last_order") > 180, "Medium")
        .otherwise("Low")
    )
    .withColumn("_computed_at", current_timestamp())
)

gold_customer_360.write.format("delta").mode("overwrite").saveAsTable("gold_customer_360")
print(f"Gold customer 360: {spark.table('gold_customer_360').count():,}")
gold_customer_360.orderBy(col("lifetime_revenue").desc()).show(10)

## 4.3 Gold: Financial Risk Dashboard

In [0]:
# Gold: FinTech Risk Dashboard
silver_wt = spark.table("silver_wire_transfers")
fraud = spark.table("bronze_fraud_signals")
accounts = spark.table("fintech_dw.accounts")

# Transaction velocity per account
txn_metrics = (silver_wt
    .groupBy("from_account")
    .agg(
        count("*").alias("total_transfers"),
        round(sum("amount"), 2).alias("total_amount_sent"),
        round(avg("amount"), 2).alias("avg_transfer"),
        round(max("amount"), 2).alias("max_transfer"),
        countDistinct("to_account").alias("unique_recipients"),
        countDistinct("transfer_date").alias("active_days")
    )
    .withColumnRenamed("from_account", "account_id")
)

# Fraud signal counts
fraud_metrics = (fraud
    .groupBy("account_id")
    .agg(
        count("*").alias("total_signals"),
        sum(when(col("severity") == "critical", 1).otherwise(0)).alias("critical_signals"),
        sum(when(col("is_confirmed") == True, 1).otherwise(0)).alias("confirmed_fraud")
    )
)

# Combine into risk dashboard
gold_risk = (txn_metrics
    .join(fraud_metrics, "account_id", "left")
    .join(accounts.select("account_id", "account_type", "risk_score", "status"), "account_id", "left")
    .na.fill(0, ["total_signals", "critical_signals", "confirmed_fraud"])
    .withColumn("composite_risk_score",
        col("risk_score") +
        (col("critical_signals") * 20) +
        (col("confirmed_fraud") * 50) +
        when(col("max_transfer") > 50000, 15).otherwise(0)
    )
    .withColumn("risk_tier",
        when(col("composite_risk_score") >= 100, "Critical")
        .when(col("composite_risk_score") >= 60, "High")
        .when(col("composite_risk_score") >= 30, "Medium")
        .otherwise("Low")
    )
    .withColumn("_computed_at", current_timestamp())
)

gold_risk.write.format("delta").mode("overwrite").saveAsTable("gold_risk_dashboard")
print(f"Gold risk dashboard: {spark.table('gold_risk_dashboard').count():,}")
gold_risk.groupBy("risk_tier").count().orderBy("count", ascending=False).show()

## Gold Layer Patterns — Multiple Gold Tables

### One Gold Table Per Use Case

Don't create ONE massive Gold table. Create FOCUSED tables for each consumer:

```
Silver Tables                    Gold Tables (one per use case)
┌──────────────┐                ┌─────────────────────────────┐
│silver_orders │───────────────▶│ gold_daily_revenue          │ → Finance dashboard
│              │                │ (date, revenue, orders)      │
│              │───────────────▶│ gold_customer_segments       │ → Marketing team
│              │                │ (customer, segment, LTV)     │
│              │───────────────▶│ gold_product_performance     │ → Product team
│              │                │ (product, sales, returns)    │
├──────────────┤                ├─────────────────────────────┤
│silver_customers│──────────────▶│ gold_churn_features         │ → ML team
│              │                │ (customer, features for ML)  │
├──────────────┤                ├─────────────────────────────┤
│silver_payments│───────────────▶│ gold_payment_reconciliation │ → Finance ops
└──────────────┘                └─────────────────────────────┘
```

### Gold Table Design Principles

| Principle | Why | Example |
|-----------|-----|---------|
| **Pre-aggregated** | Avoid repeated expensive scans | Daily revenue pre-computed |
| **Denormalized** | Analysts shouldn't need JOINs | Customer name IN the fact table |
| **Partitioned by query pattern** | Fast for common filters | Partition by date for time-series |
| **Documented** | Self-service for analysts | Column descriptions, business definitions |
| **SLA-bound** | Consumers depend on freshness | "Updated by 7 AM daily" |

In [0]:
# Build multiple Gold tables from Silver
from pyspark.sql.functions import col, count, countDistinct, sum as spark_sum, avg, round as spark_round, max as spark_max, min as spark_min, datediff, current_date, when

orders = spark.table("techmart_dw.orders")
customers = spark.table("techmart_dw.customers")

# Gold Table 1: Daily Revenue Metrics (for Finance dashboard)
print("🥇 GOLD TABLE 1: Daily Revenue Metrics")
print("=" * 60)
gold_daily = (
    orders
    .filter(col("status").isin("completed", "shipped"))
    .groupBy("order_date")
    .agg(
        count("*").alias("total_orders"),
        countDistinct("customer_id").alias("unique_customers"),
        spark_round(spark_sum("total_amount"), 2).alias("total_revenue"),
        spark_round(avg("total_amount"), 2).alias("avg_order_value"),
        spark_round(spark_sum("discount_amount"), 2).alias("total_discounts"),
    )
    .orderBy(col("order_date").desc())
)
gold_daily.show(5)

# Gold Table 2: Customer Segments (for Marketing)
print("\n🥇 GOLD TABLE 2: Customer Segments")
print("=" * 60)
gold_customers = (
    customers
    .join(
        orders.filter(col("status") == "completed")
        .groupBy("customer_id")
        .agg(
            count("*").alias("total_orders"),
            spark_sum("total_amount").alias("total_spent"),
            spark_max("order_date").alias("last_order_date"),
            spark_min("order_date").alias("first_order_date"),
        ),
        "customer_id", "left"
    )
    .withColumn("days_since_last_order", datediff(current_date(), col("last_order_date")))
    .withColumn("activity_status", 
        when(col("days_since_last_order") <= 30, "Active")
        .when(col("days_since_last_order") <= 90, "At Risk")
        .when(col("days_since_last_order") <= 180, "Lapsing")
        .otherwise("Churned")
    )
    .select("customer_id", "first_name", "last_name", "customer_segment",
            "total_orders", "total_spent", "last_order_date", 
            "days_since_last_order", "activity_status")
)
gold_customers.show(5)
print(f"Customer activity breakdown:")
gold_customers.groupBy("activity_status").count().show()


In [0]:
# ============================================
# 🎯 YOUR TURN: Build a Gold Table
# ============================================
# Create a Gold table: "Product Performance" for the Product team
#
# Columns needed:
# - product_id, product_name, category
# - total_units_sold (sum of quantity from order_items)
# - total_revenue (sum of line_total from order_items)
# - unique_customers (count distinct customer_id from orders)
# - avg_order_quantity
# - first_sold_date, last_sold_date
#
# Join: order_items → orders (for customer_id) → products (for name/category)
# Filter: only completed orders
#
# Write your Gold table query below:


In [0]:
# ============================================
# ✅ SOLUTION
# ============================================
from pyspark.sql.functions import col, count, countDistinct, sum as spark_sum, avg, min as spark_min, max as spark_max, round as spark_round

gold_products = (
    spark.table("techmart_dw.order_items").alias("oi")
    .join(spark.table("techmart_dw.orders").alias("o"), "order_id")
    .filter(col("o.status") == "completed")
    .join(spark.table("techmart_dw.products").alias("p"), "product_id")
    .groupBy("p.product_id", "p.product_name", "p.category")
    .agg(
        spark_sum("oi.quantity").alias("total_units_sold"),
        spark_round(spark_sum("oi.line_total"), 2).alias("total_revenue"),
        countDistinct("o.customer_id").alias("unique_customers"),
        spark_round(avg("oi.quantity"), 1).alias("avg_order_quantity"),
        spark_min("o.order_date").alias("first_sold_date"),
        spark_max("o.order_date").alias("last_sold_date"),
    )
    .orderBy(col("total_revenue").desc())
)

print("🥇 GOLD: Product Performance")
gold_products.show(10)
print(f"Total products with sales: {gold_products.count()}")

---
# 📗 Section 5: Incremental Processing

## Watermark Pattern

In [0]:
%sql
-- Watermark tracking table
DROP TABLE IF EXISTS pipeline_watermarks;
CREATE TABLE pipeline_watermarks (
  pipeline_name STRING, table_name STRING,
  last_processed_at TIMESTAMP, rows_processed BIGINT, updated_at TIMESTAMP
);
INSERT INTO pipeline_watermarks VALUES
  ('orders_pipeline', 'silver_orders', '2020-01-01', 0, current_timestamp()),
  ('payments_pipeline', 'silver_payments', '2020-01-01', 0, current_timestamp()),
  ('transfers_pipeline', 'silver_wire_transfers', '2023-01-01', 0, current_timestamp());

In [0]:
# Incremental load function
def incremental_load(source_table, target_table, pipeline_name, watermark_col="_ingested_at"):
    """Load only new records since last watermark."""
    # Get watermark
    wm = spark.sql(f"""
        SELECT last_processed_at FROM pipeline_watermarks
        WHERE pipeline_name = '{pipeline_name}'
    """).collect()[0].last_processed_at
    
    # Read new records
    source = spark.table(source_table)
    new_records = source.filter(col(watermark_col) > lit(wm))
    new_count = new_records.count()
    
    if new_count == 0:
        print(f"  No new records for {pipeline_name}")
        return 0
    
    # Write (append mode for incremental)
    new_records.write.format("delta").mode("append").saveAsTable(target_table)
    
    # Update watermark
    new_max = new_records.agg(max(watermark_col)).collect()[0][0]
    spark.sql(f"""
        UPDATE pipeline_watermarks
        SET last_processed_at = '{new_max}', rows_processed = rows_processed + {new_count},
            updated_at = current_timestamp()
        WHERE pipeline_name = '{pipeline_name}'
    """)
    
    print(f"  ✅ {pipeline_name}: {new_count:,} new records loaded")
    return new_count

# Demo (will show 0 since we just loaded everything)
print("Incremental load check:")
incremental_load("bronze_orders", "silver_orders", "orders_pipeline")

## Idempotent Pipelines

In [0]:
# Idempotent pattern: MERGE ensures safe re-runs
def idempotent_silver_refresh(source_table, target_table, key_col, watermark_col):
    """Safely re-runnable silver refresh using MERGE."""
    source = spark.table(source_table)
    
    # Create target if not exists
    if not spark.catalog.tableExists(target_table):
        source.limit(0).write.format("delta").saveAsTable(target_table)
    
    source.createOrReplaceTempView("_source_temp")
    
    merge_sql = f"""
        MERGE INTO {target_table} AS target
        USING _source_temp AS source
        ON target.{key_col} = source.{key_col}
        WHEN MATCHED AND source.{watermark_col} > target.{watermark_col} THEN UPDATE SET *
        WHEN NOT MATCHED THEN INSERT *
    """
    spark.sql(merge_sql)
    count = spark.table(target_table).count()
    print(f"  ✅ {target_table}: {count:,} rows (idempotent refresh)")
    return count

# This is safe to run multiple times — same result every time
print("First run:")
idempotent_silver_refresh("bronze_fraud_signals", "silver_fraud_signals", "signal_id", "_ingested_at")
print("Second run (idempotent — no change):")
idempotent_silver_refresh("bronze_fraud_signals", "silver_fraud_signals", "signal_id", "_ingested_at")

## Incremental Processing Patterns

### Pattern 1: Watermark-Based (Most Common)

```python
# Track the last processed timestamp
watermark = get_max_timestamp_from_silver()

# Only process new records
new_records = bronze.filter(col("_ingested_at") > watermark)

# MERGE into Silver
silver.merge(new_records, condition="silver.id = new.id")
```

### Pattern 2: Change Data Feed (CDF)

```python
# Read only CHANGES from a Delta table (not full table)
changes = spark.readStream.format("delta")
    .option("readChangeFeed", "true")
    .option("startingVersion", last_processed_version)
    .table("silver.orders")
```

### Pattern 3: Partition Overwrite

```python
# Overwrite only the partition being processed (idempotent!)
(new_data
    .write.format("delta")
    .mode("overwrite")
    .option("replaceWhere", f"order_date = '{process_date}'")
    .saveAsTable("gold.daily_metrics"))
```

### Choosing the Right Pattern

| Pattern | Use When | Pros | Cons |
|---------|----------|------|------|
| **Watermark** | Source has reliable timestamp | Simple, efficient | Misses late data |
| **CDF** | Need row-level changes | Complete, efficient | Requires CDF enabled |
| **Partition overwrite** | Processing by date/partition | Idempotent, simple | Full partition rewrite |
| **MERGE** | Need upsert (insert + update) | Handles all cases | More complex |

---
# 📗 Section 6: Delta Lake Features

## Time Travel

In [0]:
%sql
-- View table history
DESCRIBE HISTORY silver_orders LIMIT 5;

In [0]:
%sql
-- Time travel: query previous version
SELECT COUNT(*) AS current_count FROM silver_orders;

In [0]:
%sql
-- Query version 0 (initial state)
SELECT COUNT(*) AS v0_count FROM silver_orders VERSION AS OF 0;

## Schema Evolution

In [0]:
%sql
-- Add a new column (schema evolution)
ALTER TABLE silver_orders ADD COLUMNS (loyalty_points INT);

-- Verify
DESCRIBE silver_orders;

## OPTIMIZE and VACUUM

In [0]:
%sql
-- Compact small files
OPTIMIZE silver_orders;

In [0]:
%sql
-- View file metrics
DESCRIBE DETAIL silver_orders;

---
# 📗 Section 7: Orchestration Concepts

## Pipeline with Logging and Error Handling

In [0]:
import time
from datetime import datetime

class MedallionPipeline:
    """Orchestrated medallion pipeline with logging."""
    
    def __init__(self, name):
        self.name = name
        self.start_time = None
        self.logs = []
    
    def log(self, level, message):
        entry = f"[{datetime.now().strftime('%H:%M:%S')}] [{level}] {message}"
        self.logs.append(entry)
        print(f"  {entry}")
    
    def run_step(self, step_name, func, *args):
        """Run a pipeline step with error handling."""
        self.log("INFO", f"Starting: {step_name}")
        try:
            result = func(*args)
            self.log("INFO", f"Completed: {step_name} → {result}")
            return result
        except Exception as e:
            self.log("ERROR", f"Failed: {step_name} → {e}")
            raise
    
    def run(self, steps):
        """Execute all pipeline steps."""
        self.start_time = time.time()
        self.log("INFO", f"Pipeline '{self.name}' started")
        
        results = {}
        for step_name, func, args in steps:
            results[step_name] = self.run_step(step_name, func, *args)
        
        elapsed = time.time() - self.start_time
        self.log("INFO", f"Pipeline completed in {elapsed:.1f}s")
        return results

# Define pipeline steps
def count_table(table):
    return spark.table(table).count()

pipeline = MedallionPipeline("fintech_daily")
steps = [
    ("verify_bronze", count_table, ["bronze_wire_transfers"]),
    ("verify_silver", count_table, ["silver_wire_transfers"]),
    ("verify_gold", count_table, ["gold_risk_dashboard"]),
]
pipeline.run(steps)

---
# 🎓 Section 8: Capstone — Complete FinTech Medallion Pipeline

Build the full Bronze → Silver → Gold pipeline for all FinTech tables.

In [0]:
# CAPSTONE: Full FinTech Pipeline
from pyspark.sql.functions import *
from pyspark.sql.window import Window
import time

print("=" * 60)
print("🚀 CAPSTONE: FinTech Medallion Pipeline")
print("=" * 60)

# ---- BRONZE ----
print("\n🥉 BRONZE LAYER")
bronze_tables = {
    "accounts": "fintech_dw.accounts",
    "wire_transfers": "fintech_dw.wire_transfers",
    "fraud_signals": "fintech_dw.fraud_signals",
    "kyc_records": "fintech_dw.kyc_records",
    "compliance_events": "fintech_dw.compliance_events"
}

for name, source in bronze_tables.items():
    target = f"capstone_bronze_{name}"
    df = spark.table(source)
    bronze = (df
        .withColumn("_ingested_at", current_timestamp())
        .withColumn("_source", lit("fintech_core"))
        .withColumn("_batch", lit("capstone_batch_001"))
    )
    bronze.write.format("delta").mode("overwrite").saveAsTable(target)
    print(f"  ✅ {target}: {bronze.count():,} rows")

In [0]:
# ---- SILVER ----
print("\n🥈 SILVER LAYER")

# Silver accounts
bronze_acc = spark.table("capstone_bronze_accounts")
silver_acc = (bronze_acc
    .withColumn("account_type", lower(trim(col("account_type"))))
    .withColumn("status", lower(trim(col("status"))))
    .filter("account_id IS NOT NULL AND balance >= 0")
    .withColumn("_processed_at", current_timestamp())
    .dropDuplicates(["account_id"])
)
silver_acc.write.format("delta").mode("overwrite").saveAsTable("capstone_silver_accounts")
print(f"  ✅ capstone_silver_accounts: {silver_acc.count():,}")

# Silver wire transfers (CDC)
bronze_wt = spark.table("capstone_bronze_wire_transfers")
w = Window.partitionBy("transfer_id").orderBy(col("_cdc_timestamp").desc())
silver_wt = (bronze_wt
    .withColumn("_rn", row_number().over(w))
    .filter("_rn = 1 AND _cdc_operation != 'D'")
    .drop("_rn")
    .withColumn("status", lower(trim(col("status"))))
    .filter("transfer_id IS NOT NULL AND amount > 0")
    .withColumn("_processed_at", current_timestamp())
)
silver_wt.write.format("delta").mode("overwrite").saveAsTable("capstone_silver_transfers")
print(f"  ✅ capstone_silver_transfers: {silver_wt.count():,}")

# Silver fraud
bronze_fraud = spark.table("capstone_bronze_fraud_signals")
silver_fraud = (bronze_fraud
    .dropDuplicates(["signal_id"])
    .withColumn("severity", lower(trim(col("severity"))))
    .withColumn("signal_type", lower(trim(col("signal_type"))))
    .filter("signal_id IS NOT NULL")
    .withColumn("_processed_at", current_timestamp())
)
silver_fraud.write.format("delta").mode("overwrite").saveAsTable("capstone_silver_fraud")
print(f"  ✅ capstone_silver_fraud: {silver_fraud.count():,}")

# Silver compliance
bronze_comp = spark.table("capstone_bronze_compliance_events")
silver_comp = (bronze_comp
    .dropDuplicates(["event_id"])
    .withColumn("event_type", lower(trim(col("event_type"))))
    .withColumn("resolution_status", lower(trim(col("resolution_status"))))
    .filter("event_id IS NOT NULL AND amount_involved > 0")
    .withColumn("_processed_at", current_timestamp())
)
silver_comp.write.format("delta").mode("overwrite").saveAsTable("capstone_silver_compliance")
print(f"  ✅ capstone_silver_compliance: {silver_comp.count():,}")

In [0]:
# ---- GOLD ----
print("\n🥇 GOLD LAYER")

# Gold 1: Transaction Summary
silver_t = spark.table("capstone_silver_transfers")
gold_txn = (silver_t
    .withColumn("transfer_month", date_format("transfer_date", "yyyy-MM"))
    .groupBy("transfer_month", "currency", "status")
    .agg(
        count("*").alias("txn_count"),
        round(sum("amount"), 2).alias("total_amount"),
        round(avg("amount"), 2).alias("avg_amount"),
        countDistinct("from_account").alias("unique_senders")
    )
    .withColumn("_computed_at", current_timestamp())
)
gold_txn.write.format("delta").mode("overwrite").saveAsTable("capstone_gold_txn_summary")
print(f"  ✅ capstone_gold_txn_summary: {gold_txn.count():,}")

# Gold 2: Risk Dashboard
silver_t = spark.table("capstone_silver_transfers")
silver_f = spark.table("capstone_silver_fraud")
silver_a = spark.table("capstone_silver_accounts")

txn_agg = silver_t.groupBy(col("from_account").alias("account_id")).agg(
    count("*").alias("transfers_out"),
    round(sum("amount"), 2).alias("total_sent"),
    round(max("amount"), 2).alias("max_single_transfer")
)
fraud_agg = silver_f.groupBy("account_id").agg(
    count("*").alias("fraud_signals"),
    sum(when(col("severity") == "critical", 1).otherwise(0)).alias("critical_alerts"),
    sum(when(col("is_confirmed") == True, 1).otherwise(0)).alias("confirmed_fraud")
)

gold_risk = (silver_a
    .select("account_id", "account_type", "risk_score", "status")
    .join(txn_agg, "account_id", "left")
    .join(fraud_agg, "account_id", "left")
    .na.fill(0)
    .withColumn("risk_tier",
        when(col("confirmed_fraud") > 0, "Critical")
        .when(col("critical_alerts") > 2, "High")
        .when(col("risk_score") > 70, "Medium")
        .otherwise("Low")
    )
    .withColumn("_computed_at", current_timestamp())
)
gold_risk.write.format("delta").mode("overwrite").saveAsTable("capstone_gold_risk")
print(f"  ✅ capstone_gold_risk: {gold_risk.count():,}")

# Gold 3: Compliance Report
silver_c = spark.table("capstone_silver_compliance")
gold_compliance = (silver_c
    .groupBy("regulatory_body", "event_type", "resolution_status")
    .agg(
        count("*").alias("event_count"),
        round(sum("amount_involved"), 2).alias("total_amount"),
        min("event_date").alias("earliest"),
        max("event_date").alias("latest")
    )
    .withColumn("_computed_at", current_timestamp())
)
gold_compliance.write.format("delta").mode("overwrite").saveAsTable("capstone_gold_compliance")
print(f"  ✅ capstone_gold_compliance: {gold_compliance.count():,}")

print("\n" + "=" * 60)
print("✅ CAPSTONE COMPLETE!")
print("=" * 60)

In [0]:
%sql
-- Final verification
SELECT 'capstone_bronze_accounts' AS tbl, COUNT(*) AS rows FROM capstone_bronze_accounts
UNION ALL SELECT 'capstone_bronze_wire_transfers', COUNT(*) FROM capstone_bronze_wire_transfers
UNION ALL SELECT 'capstone_silver_accounts', COUNT(*) FROM capstone_silver_accounts
UNION ALL SELECT 'capstone_silver_transfers', COUNT(*) FROM capstone_silver_transfers
UNION ALL SELECT 'capstone_silver_fraud', COUNT(*) FROM capstone_silver_fraud
UNION ALL SELECT 'capstone_gold_txn_summary', COUNT(*) FROM capstone_gold_txn_summary
UNION ALL SELECT 'capstone_gold_risk', COUNT(*) FROM capstone_gold_risk
UNION ALL SELECT 'capstone_gold_compliance', COUNT(*) FROM capstone_gold_compliance
ORDER BY tbl;

---
# 🏆 Section 9: Interview Questions

## Q1: Explain medallion architecture and when you'd skip a layer

**Answer:**
- Bronze = raw, append-only, full audit trail
- Silver = cleaned, deduplicated, validated, business keys
- Gold = pre-aggregated, BI-ready, KPIs

**Skip Bronze:** Only for ephemeral streaming where replay isn't needed
**Skip Silver:** Only for trivial aggregations on already-clean data
**Never skip Gold:** If humans query it, they need Gold

## Q2: How do you handle late-arriving data?

**Answer:**
1. Detect: compare `event_date` vs `_ingested_at` — large gap = late
2. Reprocess: re-run Silver for affected partitions
3. Idempotent MERGE ensures no duplicates on reprocess
4. Gold tables auto-refresh from corrected Silver

## Q3: What makes a pipeline idempotent?

**Answer:** Running it N times produces the same result as running once.
- Use MERGE (not INSERT) for Silver
- Use overwrite mode for Gold aggregates
- Track watermarks to avoid reprocessing
- Hash-based deduplication catches replays

## Q4: How do you handle schema evolution?

**Answer:**
- Bronze: accept any schema (schema-on-read)
- Silver: use `mergeSchema` option for additive changes
- Breaking changes: version the table (silver_orders_v2)
- Always test schema compatibility before production deploy

## Q5: Design a CDC processing pipeline

**Answer:**
1. Bronze: ingest CDC events with operation type (I/U/D) and timestamp
2. Deduplicate: keep latest event per key (by CDC timestamp)
3. Apply: MERGE into Silver — INSERT new, UPDATE changed, soft-DELETE removed
4. Handle out-of-order: always compare timestamps before applying
5. Gold: rebuild from Silver (unaffected by CDC complexity)

---
# ✅ Notebook Complete!

**What was covered:**
- Medallion architecture theory and design principles
- Bronze: append-only ingestion with full audit trail
- Silver: deduplication, cleaning, validation, quarantine, CDC processing
- Gold: daily sales, customer 360, financial risk dashboard
- Incremental processing with watermarks
- Idempotent pipeline patterns
- Delta Lake: time travel, schema evolution, OPTIMIZE
- Orchestration with logging and error handling
- Full capstone: 5 tables through Bronze → Silver → Gold
- 5 interview questions with production-ready answers

**Databases created:** `fintech_dw` (5 source tables + medallion layers)

**Next:** Notebook 05 — SCD Masterclass

---
*All databases (techmart_dw, healthcare_dw, fintech_dw) remain available.*